# Import Required Libraries
Import necessary libraries for data manipulation and natural language processing, such as pandas and nltk or spacy.

In [1]:
import pandas as pd
import json
import os
from pathlib import Path
import re
import concurrent.futures
from tqdm.auto import tqdm

/home/s22imc10262/data/NLP/hackathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Initial Dataset
Read the raw or preliminary dataset that contains the full text data to be processed.

In [2]:
# Load the 2M recipe dataset
dataset_path = "../../data/datasets/recipe_dataset_2m.csv"
df = pd.read_csv(dataset_path)

# Drop missing values and add a synthetic 'id' since the 2M csv lacks one
df = df.dropna(subset=['title', 'ingredients', 'directions'])
df = df.reset_index(names=['id'])
df['title'] = df['title'].str.lower()

print(f"Loaded {len(df)} recipes.")

# Path to the combined image dataset
image_dataset_path = Path("../../data/datasets/combined_dataset")

Loaded 2231141 recipes.


# Define Keyword Extraction Logic
Implement functions to map image directory categories to recipe keywords.

In [3]:
def sanitize_category(category_name):
    """Convert directory names like 'spaghetti_carbonara' to 'spaghetti carbonara'."""
    return category_name.replace("_", " ").lower()

# Get all image categories
categories = [d.name for d in image_dataset_path.iterdir() if d.is_dir()]
print(f"Found {len(categories)} categories.")

# Create keyword mappings
keyword_mapping = {cat: sanitize_category(cat) for cat in categories}

Found 561 categories.


# Apply Logic and Create Keyword Dataset
Apply the extraction functions across the dataset to generate a new list containing only the keyword matched pairs.

In [6]:
import random

# Fill NA to prevent matching errors
df['title'] = df['title'].fillna('')
df['ingredients'] = df['ingredients'].fillna('')

paired_data = []

# Prepare ingredients as a single string for faster regex search if needed
print("Preparing 'ingredients_str' for regex matching...")
df['ingredients_str'] = df['ingredients'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))

# We should make the strings lowercase ONCE up front and avoid doing it per-thread!
print("Lowercasing target columns globally...")
df['title'] = df['title'].str.lower()
df['ingredients_str'] = df['ingredients_str'].str.lower()

def get_matches_for_category(category_and_keyword):
    cat, keyword = category_and_keyword
    images = list((image_dataset_path / cat).glob("*.jpg"))
    if not images:
        return []
    
    # Vectorized match skipping case-insensitivity flags to utilize fast exact string matching
    # since we've already lowercased `title` and `ingredients_str` series above!
    title_match = df['title'].str.contains(keyword, regex=False, na=False)
    ing_match = df['ingredients_str'].str.contains(keyword, regex=False, na=False)
    
    # Get all possible matches instead of just filtering to the top 5 up front!
    cat_matches = df[title_match | ing_match]
    
    if len(cat_matches) == 0:
        return []
    
    local_pairs = []
    
    for img_path in images:
        # Instead of taking the exact same first 5 recipes for every single image,
        # randomly sample 5 matches from the entire category pool per image
        num_matches = min(len(cat_matches), random.randint(5, 10))
        matches = cat_matches.sample(n=num_matches, replace=False)
        
        for idx, row in matches.iterrows():
            local_pairs.append({
                "image_path": f"data/datasets/combined_dataset/{cat}/{img_path.name}",
                "recipe_id": str(row['id']),
                "category": cat,
                "title": row['title'],
                "ingredients": row['ingredients'],
                "instructions": row['directions'] if 'directions' in row else ''
            })
    return local_pairs

print("Pre-filtering recipe pools via regex and ThreadPoolExecutor...")
categories_items = list(keyword_mapping.items())

# Using ProcessPoolExecutor instead of ThreadPoolExecutor heavily decreases Pandas/GIL blocking!
with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(get_matches_for_category, categories_items), total=len(categories_items), desc="Keyword Matching"))
    for result_list in results:
        paired_data.extend(result_list)

print(f"Generated {len(paired_data)} total pairs using keyword mapping.")

Preparing 'ingredients_str' for regex matching...
Lowercasing target columns globally...
Pre-filtering recipe pools via regex and ThreadPoolExecutor...


Keyword Matching: 100%|██████████| 561/561 [00:41<00:00, 13.37it/s]


Generated 3188535 total pairs using keyword mapping.


# Save the Keyword-Only Dataset
Export the resulting keyword-only dataset to a persistent file format such as JSON.

In [7]:
output_path = "../../data/datasets/paired_dataset_keyword_only.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(paired_data, f, indent=4)

print(f"Keyword-only dataset saved to {output_path}")

# Also save the metadata mapping for these recipes so the full recipe text can be fetched later
metadata_path = "../../data/indexes/recipe_index_metadata_keyword_only.json"
metadata_dict = {}

# Extract unique recipes used in the dataset
used_recipe_ids = set([item["recipe_id"] for item in paired_data])

for _, row in df[df['id'].astype(str).isin(used_recipe_ids)].iterrows():
    metadata_dict[str(row['id'])] = {
        'title': row['title'],
        'ingredients': row['ingredients'],
        'instructions': str(row['directions']) if 'directions' in row else ''
    }
    
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata_dict, f, indent=4)
    
print(f"Saved metadata for {len(metadata_dict)} recipes to {metadata_path}")

Keyword-only dataset saved to ../../data/datasets/paired_dataset_keyword_only.json
Saved metadata for 416909 recipes to ../../data/indexes/recipe_index_metadata_keyword_only.json
